In [ ]:
import copy

import pylab as pl
import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from dataset.geometric_wrapper import TensorGeometricModelWrapper

from its.search import InverseTransformationSearch
from search.parallel_gradient import ParallelGradientDescent
from utils.affine_transforms_old import AffineTransformation2D
from utils.sampling import BatchNegativeSampler

#torch.cuda.is_available = lambda: False
#device = torch.device("cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "modelnet10"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}



architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

train_loader_augmented = dataset_dict.get('train_loader_augmented', None)

In [ ]:
x = next(iter(test_loader_transformed))[0]

batch_size = next(iter(train_loader))[0].shape[0]

from utils.eval.vis import vis_dataset

vis_dataset(train_loader, val_loader, test_loader_transformed)
from experiment_thesis.main import train_and_get_model, train_or_load_energy_model
from experiment_thesis.dataset_preperation.basic_networks import get_network
from utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparison_augmented_models")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
dataset_augmented = train_loader_augmented.dataset

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]


In [ ]:
def evaluate_and_store_results(model, modelname,test_loader,test_loader_transformed, device, savepath):
    #check if stored results exist
    if savepath is not None and os.path.exists(savepath):
        import json
        with open(savepath, "r") as f:
            results = json.load(f)
        print(f"Loaded results for {modelname} from {savepath}:")
        print("Transformed test set results:")
        print(results["transformed_test_results"])
        print("Original test set results:")
        print(results["original_test_results"])
        return results


    res_transformed = evaluate_base_model(model, test_loader_transformed, device)
    res = evaluate_base_model(model, test_loader, device)
    print(f"Results for {modelname}:")
    print("Transformed test set results:")
    print(res_transformed)
    print("Original test set results:")
    print(res)
    if savepath is not None:
        import json
        results = {
            "modelname": modelname,
            "transformed_test_results": res_transformed,
            "original_test_results": res,
        }
        with open(savepath, "w") as f:
            json.dump(results, f, indent=4)

    return res


In [ ]:
result_folder = os.path.join(results_dir_path, transform_name)
os.makedirs(result_folder, exist_ok=True)

In [ ]:
#check main model
res = evaluate_and_store_results(model, modelname, test_loader, test_loader_transformed, device, savepath(modelname))

In [ ]:
class TensorGeometricModelUnwrapper(torch.nn.Module):
    """
    Wrapper for a torch_geometric model that receives a tuple of (pos, y) as input and creates
    a Data object from it that is passed to the model.
    """
    def __init__(self):
        super(TensorGeometricModelUnwrapper, self).__init__()

    def forward(self, data):
        # pos and y are batched tensors from a DataLoader
        # need to reconstruct the original Data objects for torch_geometric models

        pos = data.pos
        batch = data.batch
        #split pos into individual tensors based on batch
        pos_list = torch.split(pos, torch.bincount(batch).tolist())
        return torch.stack(pos_list)

In [ ]:
model_augmented = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname_augmented = f"{dataset}_{architecture}_augmented"

train_and_get_model(model_augmented,model_dir_path,modelname_augmented, train_loader_augmented, val_loader_transformed , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)

print("Augmented model results:")
res = evaluate_and_store_results(model_augmented, modelname_augmented, test_loader, test_loader_transformed, device,  savepath(modelname_augmented))

del model_augmented
torch.cuda.empty_cache()


In [ ]:
model_augmented = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname_augmented = f"{dataset}_{architecture}_augmented_val"

train_and_get_model(model_augmented,model_dir_path,modelname_augmented, train_loader_augmented, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)

res = evaluate_and_store_results(model_augmented, modelname_augmented, test_loader, test_loader_transformed, device,  savepath(modelname_augmented))
torch.cuda.empty_cache()


In [ ]:
model_augmented_single_sample = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname_augmented_single_sample = f"{dataset}_{architecture}_augmented_single_sample"
train_and_get_model(model_augmented_single_sample,model_dir_path,modelname_augmented_single_sample, train_loader_transformed, val_loader_transformed , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)

evaluate_and_store_results(model_augmented_single_sample, modelname_augmented_single_sample, test_loader, test_loader_transformed, device,  savepath(modelname_augmented_single_sample))
del model_augmented_single_sample
torch.cuda.empty_cache()

In [ ]:
if dataset=="modelnet10":
    architecture2 = "pointnetplus_euclidean"
    model_unaugmented_euclidean = get_network(dataset_info,architecture2, num_classes=n_classes).to(device)

    modelname_unaugmented_euclidean = f"{dataset}_{architecture2}_no_aug"
    train_and_get_model(model_unaugmented_euclidean,model_dir_path,modelname_unaugmented_euclidean, train_loader, val_loader , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
    },load_if_exists=True)

    print("Unaugmented Euclidean model results:")
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader_transformed, device)
    print(res)
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader, device)
    print(res)

    model_unaugmented_euclidean = None
    torch.cuda.empty_cache()

    model_augmented_euclidean = get_network(dataset_info,architecture2, num_classes=n_classes).to(device)
    modelname_augmented_euclidean = f"{dataset}_{architecture2}_augmented"
    train_and_get_model(model_augmented_euclidean,model_dir_path,modelname_augmented_euclidean, train_loader_augmented, val_loader_transformed , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
    },load_if_exists=True)

    evaluate_and_store_results(model_augmented_euclidean, modelname_augmented_euclidean, test_loader, test_loader_transformed, device,  savepath(modelname_augmented_euclidean))


In [ ]:
if dataset=="modelnet10":
    architecture2 = "pointnetplus_euclidean"
    model_unaugmented_euclidean = get_network(dataset_info,architecture2, num_classes=n_classes).to(device)

    modelname_unaugmented_euclidean = f"{dataset}_{architecture2}_no_aug"
    train_and_get_model(model_unaugmented_euclidean,model_dir_path,modelname_unaugmented_euclidean, train_loader, val_loader , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
    },load_if_exists=True)

    print("Unaugmented Euclidean model results:")
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader_transformed, device)
    print(res)
    res = evaluate_base_model(model_unaugmented_euclidean, test_loader, device)
    print(res)

    model_unaugmented_euclidean = None
    torch.cuda.empty_cache()

    model_augmented_euclidean = get_network(dataset_info,architecture2, num_classes=n_classes).to(device)
    modelname_augmented_euclidean = f"{dataset}_{architecture2}_augmented2"
    train_and_get_model(model_augmented_euclidean,model_dir_path,modelname_augmented_euclidean, train_loader_augmented, val_loader_transformed , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
    },load_if_exists=True)

    evaluate_and_store_results(model_augmented_euclidean, modelname_augmented_euclidean, test_loader, test_loader_transformed, device,  savepath(modelname_augmented_euclidean))


In [ ]:
from utils.augments import build_default_augmentations
import utils

class ComposeModel(nn.Module):
    def __init__(self, augment, model):
        super(ComposeModel, self).__init__()
        self.augment = augment
        self.model = model

    def forward(self, x):
        x = self.augment(x)
        x = self.model(x)
        return x

if is_image_data:
    affine_augment = build_default_augmentations()
    model_augmented_blur = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
    seq = ComposeModel(affine_augment, model_augmented_blur)
    modelname_augmented_blur = f"{dataset}_{architecture}_augmented_blur"
    train_and_get_model(seq,model_dir_path,modelname_augmented_blur, train_loader_augmented, val_loader_transformed , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "16-mixed",
    },load_if_exists=True)
    print("Augmented blur model results:")
    res = evaluate_and_store_results(seq, modelname_augmented_blur, test_loader, test_loader_transformed, device,  savepath(modelname_augmented_blur))




In [ ]:
import torch

def is_rotation_matrix_torch(T, tol=1e-6):
    """
    T: torch tensor of shape [N, 4, 4]
    Returns: mask of shape [N] indicating valid rotation matrices
    """
    # Extract the top-left 3x3 rotation part
    R = T[:, :3, :3]  # shape [N, 3, 3]

    # Check orthogonality: R^T * R = I
    RtR = torch.matmul(R.transpose(1, 2), R)  # [N, 3, 3]
    I = torch.eye(3, device=T.device).unsqueeze(0).expand_as(RtR)
    orthogonal_mask = torch.allclose(RtR, I, atol=tol)  # single bool for all? Not ideal, do batch:

    orthogonal_mask = torch.all(torch.abs(RtR - I) < tol, dim=(1,2))

    # Check determinant = 1
    det = torch.det(R)  # [N]
    proper_mask = torch.abs(det - 1.0) < tol

    # Combine
    valid_mask = orthogonal_mask & proper_mask

    return valid_mask, det, orthogonal_mask, proper_mask


# Example usage
T = test_loader_transformed.dataset.transformation_matrices  # shape [908,4,4]
valid_mask, dets, ortho_mask, prop_mask = is_rotation_matrix_torch(T)

print(f"Valid rotation matrices: {valid_mask.sum().item()} / {T.shape[0]}")


In [ ]:
from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images
transform_seq_test = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="individual"
).to(device)
#transform_seq_test = replace_rotation_transforms_2vec(transform_seq_test)



In [ ]:
T = transform_seq_test.initial_param(batch_size=908)
T = transform_seq_test(T)

In [ ]:
import torch
import plotly.graph_objects as go
import numpy as np

# Assume T: [908, 4, 4]
R = T[:, :3, :3]  # [N, 3, 3]

# Rotate a fixed vector (x-axis) by all rotations
v = torch.tensor([1.0, 0.0, 0.0], device=R.device).repeat(R.shape[0], 1)
rotated_vectors = torch.einsum('nij,nj->ni', R, v)
rotated_vectors = rotated_vectors / rotated_vectors.norm(dim=1, keepdim=True)

# Convert to numpy for Plotly
rotated_vectors_np = rotated_vectors.cpu().numpy()

# Sphere for reference
phi, theta = np.mgrid[0:np.pi:50j, 0:2*np.pi:50j]
x_sphere = np.sin(phi) * np.cos(theta)
y_sphere = np.sin(phi) * np.sin(theta)
z_sphere = np.cos(phi)

# Create interactive figure
fig = go.Figure()

# Add semi-transparent sphere
fig.add_trace(go.Surface(
    x=x_sphere, y=y_sphere, z=z_sphere,
    colorscale='Blues',
    opacity=0.2,
    showscale=False
))

# Add rotation vectors
fig.add_trace(go.Scatter3d(
    x=rotated_vectors_np[:,0],
    y=rotated_vectors_np[:,1],
    z=rotated_vectors_np[:,2],
    mode='markers',
    marker=dict(size=1, color='red'),
    name='Rotation vectors'
))

fig.update_layout(
    scene=dict(
        xaxis=dict(title='X', range=[-1.2,1.2]),
        yaxis=dict(title='Y', range=[-1.2,1.2]),
        zaxis=dict(title='Z', range=[-1.2,1.2]),
        aspectmode='cube'
    ),
    title='Rotation vectors on unit sphere'
)

fig.show()


In [ ]:
if is_image_data:
    import torch, gc, time
    # --- Parameter count ---
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{modelname} total parameters: {total_params:,}")
    print(f"{modelname} trainable parameters: {trainable_params:,}")

    import torchinfo

    #calculate flop count
    print(torchinfo.summary(
        model,
        input_size=(dataset_info.batch_size, *dataset_info.input_size),
        col_names=("output_size", "num_params", "mult_adds"),
        depth=3
    ))



    # --- Speed test (forward pass timing) ---
    model.eval()
    dummy_input = torch.randn(dataset_info.batch_size, *dataset_info.input_size).to(device)
    _ = model(dummy_input)
    torch.cuda.synchronize()

    torch.cuda.synchronize()
    start_time = time.time()
    with torch.no_grad():
        for _ in range(2000):  # Run multiple iterations for stable timing
            _ = model(dummy_input)
    torch.cuda.synchronize()
    elapsed = (time.time() - start_time) / 2000
    print(f"{modelname} avg forward pass time: {elapsed*1000:.2f} ms")

In [ ]:
if is_image_data:
    #speed test main model
    from experiment_thesis.dataset_preperation.basic_networks import convert_flexible_resnet_to_escnn, convert_flexible_resnet_to_gcnn
    model_escnn = convert_flexible_resnet_to_escnn(model).to(device)

    import time
    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(1000):
        outputs = model(x)
        torch.cuda.synchronize()  # Ensure all CUDA operations are complete
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 1000:.6f} seconds")

    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(1000):
        outputs = model(x)
        torch.cuda.synchronize()  # Ensure all CUDA operations are complete
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 1000:.6f} seconds")

In [ ]:
if is_image_data:
    import time
    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(100):
        outputs = model_escnn(x)
        torch.cuda.synchronize()  # Ensure all CUDA operations are complete
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 100:.6f} seconds")

    x, y = next(iter(test_loader))
    x = x.to(device)
    y = y.to(device)
    model.eval().to(device)
    start_time = time.time()
    for _ in range(100):
        outputs = model_escnn(x)
        torch.cuda.synchronize()  # Ensure all CUDA operations are complete
    end_time = time.time()
    print(f"Average inference time over 100 runs: {(end_time - start_time) / 100:.6f} seconds")

In [ ]:
model

In [ ]:
import pytorch_lightning as pl
# --- Helper to safely evaluate and delete models ---



# --- If working with image data ---
if is_image_data:
    from experiment_thesis.dataset_preperation.basic_networks import (
        convert_flexible_resnet_to_escnn,
        convert_flexible_resnet_to_gcnn,
    )

    configs = [
        ("escnn", lambda m: convert_flexible_resnet_to_escnn(m)),
        #("escnn_48", lambda m: convert_flexible_resnet_to_escnn(m,rotations=48)),
    ]
    epoch_config = {
        "escnn": dataset_info.epochs,
        "escnn_48": dataset_info.epochs//5,
    }
    preccision_config = {"escnn": "16-mixed","escnn_48":"16-mixed"}


    # Speed test and parameter count of main model
    for suffix, converter in configs:
        print(f"\n==== Training {suffix.upper()} model ====")
        modelname = f"{dataset}_{architecture}_{suffix}"
        model_tmp = converter(model).to(device)

        # # --- Parameter count ---
        total_params = sum(p.numel() for p in model_tmp.parameters())
        trainable_params = sum(p.numel() for p in model_tmp.parameters() if p.requires_grad)
        print(f"{modelname} total parameters: {total_params:,}")
        print(f"{modelname} trainable parameters: {trainable_params:,}")

        print(model_tmp)

        import torch
        import gc
        import pytorch_lightning as pl # <--- ADD THIS LINE (if it's missing)

        class gc_callback(pl.callbacks.Callback):
            def on_validation_end(self, trainer, pl_module):
                gc.collect()
                torch.cuda.empty_cache()


        # --- Train model ---
        train_and_get_model(
            model_tmp,
            model_dir_path,
            modelname,
            train_loader_augmented,
            val_loader_transformed,
            trainer_kwargs={
                "accelerator": "auto",
                "max_epochs": epoch_config[suffix],
                "precision": preccision_config[suffix],
            },
            load_if_exists=True,strict=False,
        )

        evaluate_and_store_results(model_tmp, modelname, test_loader, test_loader_transformed, device,  savepath(modelname))
        del model_tmp
        gc.collect()
        torch.cuda.empty_cache()



In [ ]:
# --- If working with ModelNet dataset ---
if dataset == "modelnet10" and True:
    model_variants = {
        "pca_then_norm_randomize": "pointnetplus_pca_then_norm_randomize",
    }

    for suffix, netname in model_variants.items():
        print(f"\n==== Training {suffix.upper()} model ====")
        model_tmp = get_network(dataset_info, netname, num_classes=n_classes).to(device)
        modelname = f"{dataset}_{architecture}_{suffix}"

        # --- Parameter count ---
        total_params = sum(p.numel() for p in model_tmp.parameters())
        trainable_params = sum(p.numel() for p in model_tmp.parameters() if p.requires_grad)
        print(f"{modelname} total parameters: {total_params:,}")
        print(f"{modelname} trainable parameters: {trainable_params:,}")



        # --- Train model ---
        train_and_get_model(
            model_tmp,
            model_dir_path,
            modelname,
            train_loader_augmented,
            val_loader_transformed,
            trainer_kwargs={
                "accelerator": "auto",
                "max_epochs": dataset_info.epochs,
                "precision": "32",
            },
            load_if_exists=True,
        )

        evaluate_and_store_results(model_tmp, modelname, test_loader, test_loader_transformed, device,  savepath(modelname))
        del model_tmp
        import gc
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
# --- If working with ModelNet dataset ---
if dataset == "modelnet10":
    model_variants = {
        "pca_then_norm_randomize_no_aug": "pointnetplus_pca_then_norm_randomize",
        "pca_then_norm_randomize_sorted_no_aug": "pointnetplus_pca_then_norm_randomize_sort",
        "pointnetplus_euclidean_pca": "pointnetplus_pca_then_norm_randomize_euclidean",
        "pointnetplus_euclidean_pca2": "pointnetplus_pca_then_norm_randomize_euclidean",
        "pointnetplus_euclidean_pca3": "pointnetplus_pca_then_norm_randomize_euclidean"

    }

    for suffix, netname in model_variants.items():
        print(f"\n==== Training {suffix.upper()} model ====")
        model_tmp = get_network(dataset_info, netname, num_classes=n_classes).to(device)
        modelname = f"{dataset}_{architecture}_{suffix}"
        train_and_get_model(
            model_tmp,
            model_dir_path,
            modelname,
            train_loader,
            val_loader,
            trainer_kwargs={
                "accelerator": "auto",
                "max_epochs": dataset_info.epochs,
                "precision": "32",
            },
            load_if_exists=True,
        )
        evaluate_and_store_results(model_tmp, modelname, test_loader, test_loader_transformed, device,  savepath(modelname))
        del model_tmp
        import gc
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def visualize_point_cloud(pc, title="Point Cloud"):
    """
    pc: tensor of shape (N,3) or (1,N,3) or (B,N,3)
    """
    if pc.dim() == 3:
        pc = pc[0]  # take first sample in batch

    pc = pc.detach().cpu().numpy()

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(pc[:,0], pc[:,1], pc[:,2], s=3)
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    plt.show()


In [ ]:
import torch.nn as nn

def extract_preprocessing_wrapper(model_wrapper):
    """
    Extract all modules in model_wrapper.model BEFORE PointNetPlus,
    and return them wrapped in a new TensorGeometricModelWrapper.

    Args:
        model_wrapper: the original TensorGeometricModelWrapper

    Returns:
        TensorGeometricModelWrapper with only preprocessing modules
    """

    seq = model_wrapper.model
    if not isinstance(seq, nn.Sequential):
        raise ValueError("Expected model_wrapper.model to be nn.Sequential")

    preprocessing_layers = []
    for layer in seq:
        # stop when reaching PointNetPlus
        if layer.__class__.__name__ == "PointNetPlus":
            break
        preprocessing_layers.append(layer)

    # create a new Sequential with the extracted layers
    preproc_seq = nn.Sequential(*preprocessing_layers)

    # rewrap in same wrapper class
    return type(model_wrapper)(model=preproc_seq)

if dataset=="modelnet10":
    suffix =  "pca_then_norm_randomize_sorted_no_aug"
    model_tmp = get_network(dataset_info, "pointnetplus_pca_then_norm_randomize_sort", num_classes=n_classes).to(device)

    modelname = f"{dataset}_{architecture}_{suffix}"
    train_and_get_model(
        model_tmp,
        model_dir_path,
        modelname,
        train_loader,
        val_loader,
        trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
        },
        load_if_exists=True,
    )
    model_tmp


    preprocess = extract_preprocessing_wrapper(model_tmp)
    preprocess = torch.nn.Sequential(preprocess,TensorGeometricModelUnwrapper())
    preprocess.train()



    #visualize prepreoces
    x = next(iter(test_loader_transformed))[0]
    x_2 = next(iter(test_loader))[0]
    x = x.to(device)

    x_preprocessed = preprocess(x)
    x_2 = x_2.to(device)
    x_2_preprocessed = preprocess(x_2)

    #visualize point clouds

    # visualize original
    visualize_point_cloud(x[[2]], title="Original Point Cloud")

    # visualize preprocessed
    visualize_point_cloud(x_preprocessed[[2]], title="Preprocessed Point Cloud")
    visualize_point_cloud(x_2_preprocessed[[2]], title="Preprocessed Point Cloud")

In [ ]:
if dataset=="modelnet10":
    import torch

    import torch

    def collect_preprocessing_variants_vectorized(
            preprocess,
            batch,
            target_variants=4,
            max_repeats=500000,
            atol=1e-6,
    ):
        """
        Vectorized collection of unique preprocessing outputs per sample.

        Args:
            preprocess: preprocessing model
            batch: (B, N, C) tensor
            target_variants: number of variants to collect per sample
            max_repeats: safety upper bound
            atol: equality tolerance

        Returns:
            variants: tensor of shape (B, target_variants, N, C)
        """

        B, N, C = batch.shape

        # storage for variants (start empty)
        # fill with NaNs so "unused" slots never match
        variants = torch.full(
            (B, target_variants, N, C),
            float('nan'),
            dtype=batch.dtype,
            device='cpu'
        )

        # count how many variants each sample has
        counts = torch.zeros(B, dtype=torch.long)

        for _ in range(max_repeats):

            out = preprocess(batch).detach().cpu()              # (B, N, C)
            out_expanded = out[:, None, :, :]                   # (B, 1, N, C)

            # gather current variants used slots
            used = counts[:, None, None, None] > torch.arange(target_variants)[None, :, None, None]

            # compare out to existing variants (broadcast)
            # (B, K, N, C) difference
            diffs = torch.abs(out_expanded - variants)
            max_diffs = diffs.max(dim=-1).values.max(dim=-1).values  # (B, K)

            # determine if equal to *any* existing variant
            is_equal = (max_diffs <= atol) & used.squeeze(-1).squeeze(-1)  # (B, K)
            has_match = is_equal.any(dim=1)                                # (B)

            # mask for samples that need new variant
            need_new = ~has_match & (counts < target_variants)

            # if nothing to add, continue
            if need_new.sum() == 0:
                continue

            # indices to write to: where counts point
            write_positions = counts.clone()

            # write out only to samples needing new variant
            variants[need_new, write_positions[need_new]] = out[need_new]

            # increment counts
            counts[need_new] += 1

            # stop if all samples reached goal
            if (counts == target_variants).all():
                break

        #if not broken print how many variants missing
        if (counts == target_variants).all():
            return variants
        else:
            print(counts)
            return variants


    batch = next(iter(test_loader))[0].to(device)

    variants = collect_preprocessing_variants_vectorized(
        preprocess,
        batch,
        target_variants=4,
        max_repeats=1000,
        atol=1e-4
    )




In [ ]:
if dataset=="modelnet10":
    import torch
    import matplotlib.pyplot as plt

    # ---------------------------------------------
    # Reuse your existing functions
    # ---------------------------------------------

    def visualize_point_cloud(pc, title="Point Cloud"):
        """
        pc: tensor of shape (B, N, 3) or (N, 3)
        """
        if pc.dim() == 3:
            pc = pc[0]  # take first sample if batched

        pc = pc.detach().cpu().numpy()
        fig = plt.figure(figsize=(6,6))
        ax = fig.add_subplot(111, projection="3d")
        ax.scatter(pc[:,0], pc[:,1], pc[:,2], s=3)
        ax.set_title(title)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        plt.show()


    def variants_match(v_raw, v_trans, atol=1e-4,rtol=1e-4):
        """
        Check if per-sample variant sets match.
        Order does not matter.
        """
        B, K, N, C = v_raw.shape

        # Flatten each variant per sample for easier comparison
        flat_raw = v_raw.reshape(B, K, -1)
        flat_trans = v_trans.reshape(B, K, -1)

        matches = torch.zeros(B, dtype=torch.bool)

        for i in range(B):
            used = torch.zeros(K, dtype=torch.bool)
            sample_ok = True

            for kr in range(K):
                r = flat_raw[i, kr]
                found_match = False
                for kt in range(K):
                    if used[kt]:
                        continue
                    if torch.allclose(r, flat_trans[i, kt], atol=atol,rtol=rtol):
                        used[kt] = True
                        found_match = True
                        break
                if not found_match:
                    sample_ok = False
                    break

            matches[i] = sample_ok

        return matches

    # ---------------------------------------------
    # Main evaluation and visualization loop
    # ---------------------------------------------
    for (raw_batch, _), (aug_batch, _) in zip(test_loader, test_loader_transformed):

        raw_batch = raw_batch.to(device)
        aug_batch = aug_batch.to(device)

        # Collect 4 variants per sample
        v_raw = collect_preprocessing_variants_vectorized(
            preprocess,
            raw_batch,
            target_variants=4,
            max_repeats=5000,
            atol=1e-4,
        )

        v_trans = collect_preprocessing_variants_vectorized(
            preprocess,
            aug_batch,
            target_variants=4,
            max_repeats=5000,
            atol=1e-4
        )

        matches = variants_match(v_raw, v_trans,atol=2e-5,rtol=1e-5)
        print("Matches per sample:", matches.tolist())

        if not matches.all():
            bad_samples = (~matches).nonzero().flatten().tolist()
            print("Mismatch for samples:", bad_samples)

            # visualize mismatched samples
            for sample_idx in bad_samples:
                print(f"\nSample {sample_idx} - Raw variants")
                for k in range(4):
                    visualize_point_cloud(v_raw[sample_idx, k].unsqueeze(0),
                                          title=f"Raw Sample {sample_idx} - Variant {k}")

                print(f"\nSample {sample_idx} - Transformed variants")
                for k in range(4):
                    visualize_point_cloud(v_trans[sample_idx, k].unsqueeze(0),
                                          title=f"Transformed Sample {sample_idx} - Variant {k}")
                break
            break


In [ ]:
if dataset=="modelnet10":
    import torch
    import matplotlib.pyplot as plt

    def visualize_point_cloud_difference(raw_pc, trans_pc, title="Point Cloud Diff", atol=1e-6):
        """
        Visualize point cloud differences between raw and transformed variants.
        Points that differ are marked red; matching points are gray.

        Args:
            raw_pc: (N,3) torch tensor
            trans_pc: (N,3) torch tensor
            atol: tolerance for considering points equal
        """
        raw_pc = raw_pc.detach().cpu().numpy()
        trans_pc = trans_pc.detach().cpu().numpy()

        # compute per-point difference
        diff_mask = (abs(raw_pc - trans_pc) > atol).any(axis=1)  # True where difference > atol

        fig = plt.figure(figsize=(6,6))
        ax = fig.add_subplot(111, projection='3d')

        # plot matching points
        ax.scatter(raw_pc[~diff_mask,0], raw_pc[~diff_mask,1], raw_pc[~diff_mask,2],
                   c='gray', s=5, label='matching')

        # plot differing points
        ax.scatter(raw_pc[diff_mask,0], raw_pc[diff_mask,1], raw_pc[diff_mask,2],
                   c='red', s=10, label='different')

        ax.set_title(title)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        ax.legend()
        plt.show()

    def find_closest_variant(raw_variant, trans_variants, atol=1e-6):
        """
        Find the variant in trans_variants closest to raw_variant
        based on maximum absolute difference per point.
        """
        min_diff = None
        best_idx = 0
        for idx in range(trans_variants.shape[0]):
            diff = (abs(raw_variant - trans_variants[idx])).max()
            if (min_diff is None) or (diff < min_diff):
                min_diff = diff
                best_idx = idx
        return best_idx

    # ------------------------------------------------
    # Updated loop for mismatched samples
    # ------------------------------------------------
    for (raw_batch, _), (aug_batch, _) in zip(test_loader, test_loader_transformed):
        raw_batch = raw_batch.to(device)
        aug_batch = aug_batch.to(device)

        v_raw = collect_preprocessing_variants_vectorized(preprocess, raw_batch, target_variants=4, max_repeats=5000, atol=1e-4)
        v_trans = collect_preprocessing_variants_vectorized(preprocess, aug_batch, target_variants=4, max_repeats=5000, atol=1e-4)

        matches = variants_match(v_raw, v_trans,atol=1e-5,rtol=1e-4)
        print("Matches per sample:", matches.tolist())

        if not matches.all():
            bad_samples = (~matches).nonzero().flatten().tolist()
            print("Mismatch for samples:", bad_samples)

            for sample_idx in bad_samples:
                print(f"\nSample {sample_idx} - Differences highlighted in red")

                for k in range(4):
                    raw_var = v_raw[sample_idx, k]
                    # find closest variant in transformed set
                    best_idx = find_closest_variant(raw_var, v_trans[sample_idx])
                    trans_var = v_trans[sample_idx, best_idx]

                    visualize_point_cloud_difference(raw_var, trans_var,
                                                     title=f"Sample {sample_idx} - Raw Variant {k} vs Trans Variant {best_idx}")
                break
            break


In [ ]:
if dataset=="tu_berlin":
    arhcitecture_tu_berlin_invariant = "bi_lstm_pca"
    model_tu_berlin_invariant = get_network(dataset_info,arhcitecture_tu_berlin_invariant, num_classes=n_classes).to(device)
    modelname_tu_berlin_invariant = f"{dataset}_{arhcitecture_tu_berlin_invariant}"
    train_and_get_model(model_tu_berlin_invariant,model_dir_path,modelname_tu_berlin_invariant, train_loader_augmented, val_loader_transformed , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs,
            "precision": "32",
    },load_if_exists=True)

    evaluate_and_store_results(model_tu_berlin_invariant, modelname_tu_berlin_invariant, test_loader, test_loader_transformed, device,  savepath(modelname_tu_berlin_invariant))
    del model_tu_berlin_invariant
    torch.cuda.empty_cache()



In [ ]:



from utils.transforms.apply import grid_resample
from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

from utils.replacer import replace_rotation_transforms_2vec

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:
print(transform_seq.transformations)

In [ ]:
from experiment_thesis.dataset_preperation.basic_networks import make_deterministic
make_deterministic(model)



In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_layer_embedding_cache_config,create_layer_embedding_cache
cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
detectors = ["knn","per_class_knn", "knn_mixed","per_class_knn_mixed","knn_mixed_faiss","knn_itf","vim","react","dice","ash","she","laplace_mi","laplace_energy","laplace_weighted","trust_score","openmax","mahalanobis","rmd","class_prototype","react_all","energy","per_class_prototype","single_mahalanobis","single_rmd","single_mahalanobis_individual","single_rmd_individual","mahalanobis_individual","rmd_individual","prototype_multi", "laplace_entropy","adjusted_entropy","nn_guided","nn_guided_one"]

In [ ]:
# python
import os
import json
import math
import torch
from typing import Tuple, Optional, Any, Dict

from experiment_thesis.ood.base_prepare import create_ood_problem, get_default_ood_params

def find_best_detector_and_instantiate(
    base_results_dir: str,
    detectors: list,
    model,
    train_cache,
    transform_seq_arg,
    dataset_info,
    architecture: str,
    device,
    val_id_loader,
    val_ood_loader,
    prefer_second: Optional[str] = None,
) -> Tuple[
    Optional[str],
    Optional[Any],
    Optional[Dict[str, float]],
    Optional[str],
    Optional[Any],
    Optional[Dict[str, float]],
]:
    """
    Find best detector by recorded `accuracy_mean` (preferring mean) and instantiate its OOD problem.
    Returns:
      (best_detector, best_problem, best_score_dict, second_choice, second_problem, second_score_dict)
    where score dict is `{"mean": float, "se": float}` (values may be math.nan if missing).
    """

    def _load_score(det_name: str) -> Optional[Dict[str, float]]:
        eval_path = os.path.join(base_results_dir, det_name, "eval_results.json")
        eval_default_path = os.path.join(base_results_dir, det_name, "eval_results_default.json")

        for p in (eval_path, eval_default_path):
            if os.path.exists(p):
                try:
                    with open(p, "r") as f:
                        data = json.load(f)
                except Exception:
                    continue
                mean = data.get("accuracy_mean")
                se = data.get("accuracy_se", data.get("accuracy_std"))  # prefer se, fall back to std
                # normalize types
                mean_val = float(mean) if isinstance(mean, (int, float)) else math.nan
                se_val = float(se) if isinstance(se, (int, float)) else math.nan
                if not math.isnan(mean_val):
                    return {"mean": mean_val, "se": se_val}
        return None

    # 1) scan detectors for optimized accuracy (use mean)
    best_detector: Optional[str] = None
    best_score_val = -math.inf
    best_score_dict: Optional[Dict[str, float]] = None
    for det in detectors:
        score_dict = _load_score(det)
        if score_dict is None:
            continue
        mean = score_dict["mean"]
        if mean > best_score_val:
            best_score_val = mean
            best_detector = det
            best_score_dict = score_dict

    # If we didn't find any valid scored detector, return Nones
    if best_detector is None:
        return None, None, None, None, None, None

    # 2) load best params for best_detector (or default if no params file)
    det_dir = os.path.join(base_results_dir, best_detector)
    params_path = os.path.join(det_dir, "best_params.json")
    best_params = None
    if os.path.exists(params_path):
        try:
            with open(params_path, "r") as f:
                best_params = json.load(f)
        except Exception:
            best_params = None

    if best_params is None:
        best_params = get_default_ood_params(best_detector)

    # 3) try load saved model states for this detector (if any)
    best_model_params = []
    prefix = os.path.join(det_dir, "best_model")
    i = 0
    while os.path.exists(f"{prefix}_{i}.pt"):
        try:
            best_model_params.append(torch.load(f"{prefix}_{i}.pt", map_location="cpu"))
        except Exception:
            pass
        i += 1

    final_kwargs = {
        "model": model,
        "train_cache": train_cache,
        "transform_seq": transform_seq_arg,
        "dataset_info": dataset_info,
        "architecture": architecture,
        "device": str(device),
        "val_id_loader": val_id_loader,
        "val_ood_loader": val_ood_loader,
    }
    if best_model_params:
        final_kwargs["model_params"] = best_model_params

    best_problem = create_ood_problem(best_detector, best_params, **final_kwargs)

    # 4) prepare the second problem: choose the better of energy vs entropy by recorded performance
    if prefer_second and prefer_second in detectors:
        second_choice = prefer_second
        second_score_dict = _load_score(second_choice)
    else:
        candidates = [d for d in ("energy", "entropy") if d in detectors]
        second_choice = None
        best_cand_val = -math.inf
        second_score_dict = None
        for c in candidates:
            sdict = _load_score(c)
            if sdict is not None and sdict["mean"] > best_cand_val:
                best_cand_val = sdict["mean"]
                second_choice = c
                second_score_dict = sdict

    # avoid selecting same detector twice
    if second_choice == best_detector:
        other = "entropy" if best_detector == "energy" else "energy"
        if other in detectors:
            other_score = _load_score(other)
            if other_score is not None:
                second_choice = other
                second_score_dict = other_score
            else:
                second_choice = None
                second_score_dict = None
        else:
            second_choice = None
            second_score_dict = None

    if second_choice is None:
        second_problem = None
    else:
        det2_dir = os.path.join(base_results_dir, second_choice)
        params2_path = os.path.join(det2_dir, "best_params.json")
        params2 = None
        if os.path.exists(params2_path):
            try:
                with open(params2_path, "r") as f:
                    params2 = json.load(f)
            except Exception:
                params2 = None
        if params2 is None:
            params2 = get_default_ood_params(second_choice)

        model_params2 = []
        prefix2 = os.path.join(det2_dir, "best_model")
        j = 0
        while os.path.exists(f"{prefix2}_{j}.pt"):
            try:
                model_params2.append(torch.load(f"{prefix2}_{j}.pt", map_location="cpu"))
            except Exception:
                pass
            j += 1

        final_kwargs2 = dict(final_kwargs)
        if model_params2:
            final_kwargs2["model_params"] = model_params2

        second_problem = create_ood_problem(second_choice, params2, **final_kwargs2)

    return best_detector, best_problem, best_score_dict, second_choice, second_problem, second_score_dict

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "results",
    "comparison_unsupervised",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)

best_detector, best_problem, best_score, second_choice, second_problem, second_score = find_best_detector_and_instantiate(
    base_results_dir=base_results_dir,
    detectors=detectors,
    model=model,
    train_cache=train_cache,
    transform_seq_arg=transform_seq,
    dataset_info=dataset_info,
    architecture=architecture,
    device=device,
    val_id_loader=val_loader_transformed,
    val_ood_loader=val_loader_transformed,
)
#apply best problem to test loader
#apply best problem to test loader
save_path_best = os.path.join(
    base_results_dir,
    best_detector,
    "test_results.json"  # <--- This is the suggested save path
)
from search.shgo import SHGO
optimizer = SHGO(initial_samples=46, local_runs=2, local_max_steps=3,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer = SHGO(initial_samples=60,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})



from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=best_problem,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search,  save_path=save_path_best,repeats=5)

In [ ]:
modelname = f"{dataset}_{architecture}"

In [ ]:
import os
import json
import math
from typing import Optional, Dict, Any, List, Tuple


def _load_score(base_results_dir: str, det_name: str) -> Optional[Dict[str, float]]:
    """Helper to load evaluation scores (mean/se) for a detector from its results file."""
    eval_path = os.path.join(base_results_dir, det_name, "eval_results.json")
    eval_default_path = os.path.join(base_results_dir, det_name, "eval_results_default.json")

    for p in (eval_path, eval_default_path):
        if os.path.exists(p):
            try:
                with open(p, "r") as f:
                    data = json.load(f)
            except Exception:
                continue
            mean = data.get("accuracy_mean")
            se = data.get("accuracy_se", data.get("accuracy_std"))

            # Normalize types
            mean_val = float(mean) if isinstance(mean, (int, float)) else math.nan
            se_val = float(se) if isinstance(se, (int, float)) else math.nan

            if not math.isnan(mean_val):
                return {"mean": mean_val, "se": se_val}
    return None


def load_best_detector_scores_by_components(
    current_path: str,
    detectors: List[str],
    dataset_info: Any,
    architecture: str,
    dataset_name: str,
    test_filename: str = "test_results.json",
    score_key: str = "accuracy_mean"
) -> Tuple[
    Optional[str],     # Best Detector Name
    Optional[float],   # Final Test Score (from test_filename)
    Optional[float]    # Optimized Selection Score (from _load_score)
]:
    """
    Constructs the base results directory, finds the detector with the best
    recorded 'accuracy_mean' (Optimized Selection Score), and loads its
    Optimized Selection Score and final test score.
    """

    # 1. Reconstruct the base_results_dir path
    transform_seq_name = getattr(dataset_info, "transform_seq_name", "default")

    base_results_dir = os.path.join(
        current_path,
        "experiment_files",
        "results",
        "comparison_unsupervised",
        str(dataset_name),
        str(architecture),
        transform_seq_name,
    )

    # 2. Find the best detector and its Optimized Selection Score
    best_detector: Optional[str] = None
    best_score_val = -math.inf
    optimized_selection_score: Optional[float] = None # Renamed variable

    for det in detectors:
        score_dict = _load_score(base_results_dir, det)
        if score_dict is None:
            continue
        mean = score_dict["mean"]

        if mean > best_score_val:
            best_score_val = mean
            best_detector = det
            # Store the score used for selection
            optimized_selection_score = mean # Storing the score

    if best_detector is None:
        print("No valid scored detector found.")
        return None, None, None

    print(f"\nBest Detector Found: **{best_detector}** (Optimized Selection Score: {optimized_selection_score:.4f})")
    det_dir = os.path.join(base_results_dir, best_detector)

    # 3. Load Test Results and extract the final score
    final_test_score: Optional[float] = None # Renamed variable
    results_path = os.path.join(det_dir, test_filename)

    if os.path.exists(results_path):
        try:
            with open(results_path, "r") as f:
                test_results = json.load(f)

                # Extract the final test score
                test_score = test_results.get(score_key)
                if isinstance(test_score, (int, float)):
                    final_test_score = float(test_score)
                    print(f"Loaded **Final Test Score** ({score_key}): {final_test_score:.4f}")
                else:
                    print(f"Test results found, but could not extract score key: '{score_key}'")

        except Exception as e:
            print(f"Error loading final test results: {e}")
    else:
        print(f"Final Test results file **not found** at: {results_path}")

    # Return: name, final_test_score, optimized_selection_score
    return best_detector, final_test_score, optimized_selection_score

In [ ]:
load_best_detector_scores_by_components(current_path,detectors,dataset_info,architecture,dataset)


In [ ]:
from search.shgo import SHGO

optimizer_120 = SHGO(initial_samples=120-27, local_runs=3, local_max_steps=4,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer_120 = SHGO(initial_samples=120,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})

In [ ]:

from utils.augments import small_affine_augment_2d, ComposeAugmentations, random_blur_or_sharpen
from utils.sampling_strategy import TransformLatentSamplingStrategy
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from utils.transformation_problem import TransformationProblem

architecture_half = architecture+"_half"

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None

def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


energy_model2_half = get_network(dataset_info,architecture_half, num_classes=1).to(device)
from experiment_thesis.main import train_or_load_energy_model
negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    = transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
energy_conf_half = train_or_load_energy_model(
    energy_model2_half, model_dir_path, f"{modelname}_energy2_half", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_half.to(device).eval()
problem_energy_half = TransformationProblem(energy_conf_half, transform_seq,                                              consolidate_method="consolidate_simple")

def savepath_later_layer(label: str) -> str:
    model_dir_path = os.path.join(current_path, "experiment_files", "models")
    embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
    # Add results dir and helper for save paths
    results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods")
    os.makedirs(results_dir_path, exist_ok=True)

    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

res_120 = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath_later_layer("learned_energy_confidence_half_transformed_120"), show_progress=True,
    repeats=5)

res_120_real_data = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer_120, problem=problem_energy_half,
    test_loader=test_loader, max_batch_override=dataset_info.batch_size_search, show_progress=True, save_path=
    savepath_later_layer("learned_energy_confidence_half_transformed_120_untransformed"),
    repeats=5)



del energy_model2_half
del energy_conf_half
del problem_energy_half

print(res_120["accuracy_mean"])






In [ ]:
import os
# Assuming these are available in the execution environment
# from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
# from utils.transformation_problem import TransformationProblem
# import torch # Required by the objects passed in (model, problem, etc.)

def load_energy_learned(
    dataset_info,
    transform_name,
    current_path,
    dataset,
    architecture
):
    """
    Loads pre-computed evaluation results for the 'learned_energy_confidence_half_transformed_120'
    variants.

    Note: This function assumes that 'load_or_run_evaluate_confidence_and_search'
    supports a 'load_only=True' flag, which is set here to ensure it only loads
    the results from the save_path and does not re-run the evaluation.
    """

    def _savepath_later_layer(label: str) -> str:
        """Utility function to calculate the standardized save path for results."""
        # Add results dir and helper for save paths
        results_dir_path = os.path.join(
            current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods"
        )
        # Ensure the base directory exists before attempting to join with transform_name
        os.makedirs(results_dir_path, exist_ok=True)

        safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
        # Use transform_name as a subdirectory
        return os.path.join(results_dir_path, transform_name, f"{safe}.json")

    # --- 1. Load Transformed Data Result ---
    save_path_transformed = _savepath_later_layer(
        "learned_energy_confidence_half_transformed_120"
    )
    try:
        with open(save_path_transformed, "r") as f:
            res_120 = json.load(f)
    except:
        res_120 = None
    # --- 2. Load Untransformed Data Result ---
    save_path_untransformed = _savepath_later_layer(
     "learned_energy_confidence_half_transformed_120_untransformed"
    )
    try:
        with open(save_path_untransformed, "r") as f:
            res_120_real_data = json.load(f)
    except:
        res_120_real_data = None

    return res_120_real_data, res_120

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load all results from the experiment
result_folder = os.path.join(results_dir_path, transform_name)

# Define model names to look for
model_names = [
    f"{dataset}_{architecture}",  # Base model
    f"{dataset}_{architecture}_augmented",  # Augmented
    f"{dataset}_{architecture}_augmented_val",  # Augmented with val
    f"{dataset}_{architecture}_augmented_single_sample",  # Single sample
]

# Add architecture-specific models
if is_image_data:
    model_names.extend([
        f"{dataset}_{architecture}_escnn",
        f"{dataset}_{architecture}_escnn_48",
    ])

if dataset == "modelnet10":
    model_names.extend([
        f"{dataset}_{architecture}_pca_then_norm_randomize_no_aug",
        f"{dataset}_{architecture}_pca_then_norm_randomize_sorted_no_aug",
        f"{dataset}_{architecture}_pointnetplus_euclidean_pca",
        f"{dataset}_{architecture2}_no_aug",
        f"{dataset}_{architecture2}_augmented",
    ])

if dataset == "tu_berlin":
    model_names.append(f"{dataset}_{architecture}_pca")

# Load results
results = {}
for model_name in model_names:
    result_path = savepath(model_name)
    if os.path.exists(result_path):
        try:
            with open(result_path, 'r') as f:
                results[model_name] = json.load(f)
        except Exception as e:
            print(f"Error loading {model_name}: {e}")

# Extract data for plotting
model_labels = []
original_acc = []
transformed_acc = []

for model_name, result in results.items():
    # Simplify label
    label = model_name.replace(f"{dataset}_{architecture}_", "").replace(f"{dataset}_", "")
    model_labels.append(label)

    # Get accuracies
    orig = result.get('original_test_results', {}).get('accuracy', 0) * 100
    trans = result.get('transformed_test_results', {}).get('accuracy', 0) * 100

    original_acc.append(orig)
    transformed_acc.append(trans)

# Create bar plot
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(model_labels))
width = 0.35

bars1 = ax.bar(x - width/2, original_acc, width, label='Original Test', alpha=0.8)
bars2 = ax.bar(x + width/2, transformed_acc, width, label='Transformed Test', alpha=0.8)

# Customize plot
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title(f'Model Performance Comparison - {dataset.upper()} ({transform_name})',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim([0, 105])

# Add value labels on bars
def autolabel(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}',
                ha='center', va='bottom', fontsize=8)

autolabel(bars1)
autolabel(bars2)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*80)
print(f"RESULTS SUMMARY FOR {dataset.upper()} - {transform_name}")
print("="*80)
for i, label in enumerate(model_labels):
    print(f"{label:40s} | Original: {original_acc[i]:5.2f}% | Transformed: {transformed_acc[i]:5.2f}% | Gap: {abs(original_acc[i]-transformed_acc[i]):5.2f}%")
print("="*80)

In [ ]:
import json
import os
from pathlib import Path

# Define all datasets to process
all_datasets = ["bigger_mnist","bigger_emnist", "coil100", "tu_berlin", "modelnet10"]



def load_result(dataset_name, architecture_name, transform_name, model_suffix):
    """Load results for a specific model configuration."""
    base_path = os.path.join(
        current_path, "experiment_files", "results", dataset_name,
        architecture_name, "comparison_augmented_models", transform_name
    )

    model_name = f"{dataset_name}_{architecture_name}_{model_suffix}" if model_suffix else f"{dataset_name}_{architecture_name}"
    safe_name = "".join(c if c.isalnum() or c in "-_." else "_" for c in model_name)
    result_path = os.path.join(base_path, f"{safe_name}.json")

    if os.path.exists(result_path):
        with open(result_path, 'r') as f:
                data = json.load(f)
                orig_acc = data.get('original_test_results', {}).get('accuracy', None)
                trans_acc = data.get('transformed_test_results', {}).get('accuracy', None)
                print("ds"+str(dataset_name)+str(architecture_name)+str(model_suffix))
                return orig_acc, trans_acc
    return None, None

def format_accuracy(orig, trans):
    """Format accuracy values for LaTeX table."""
    if orig is None or trans is None:
        return "---", "---"
    return f"{orig*100:.1f}", f"{trans*100:.1f}"



# Generate LaTeX table
latex_lines = []

# Table 1: Original Test Set
latex_lines.append("\\begin{table}[htbp]")
latex_lines.append("\\centering")
latex_lines.append("\\caption{Accuracy on Original Test Set (\\%)}")
latex_lines.append("\\label{tab:original_test}")
latex_lines.append("\\begin{tabular}{l|cccccc}")
latex_lines.append("\\toprule")
latex_lines.append("Dataset & Single & Augmented & Aug+Val & Invariant & Energy & Best Unsup. \\\\")
latex_lines.append("\\midrule")

for ds in all_datasets:
    arch = default_architecutre_mapping[ds]
    dataset_info22 = get_dataset_info(ds)
    trans_name = dataset_info22.transform_seq_name

    orig_single, trans_single = load_result(ds, arch, trans_name, "augmented_single_sample")
    orig_aug, trans_aug = load_result(ds, arch, trans_name, "augmented")
    orig_aug_val, trans_aug_val = load_result(ds, arch, trans_name, "augmented_val")

    if ds in ["mnist", "bigger_mnist", "emnist", "bigger_emnist", "coil100"]:
        orig_inv, trans_inv = load_result(ds, arch, trans_name, "escnn")
    elif ds == "tu_berlin":
        orig_inv, trans_inv = load_result(ds, arch, trans_name, "pca")
    elif ds == "modelnet10":
        orig_inv, trans_inv = load_result(ds, arch, trans_name, "pca_then_norm_randomize_sorted_no_aug")
    else:
        orig_inv, trans_inv = None, None

    orig_energy, trans_energy = load_energy_learned(dataset_info22, trans_name, current_path, ds, arch)
    if orig_energy is not None:
        orig_energy = orig_energy["accuracy_mean"]
        trans_energy = trans_energy["accuracy_mean"]

    unsupervised_best_name, orig_unsup, trans_unsup = load_best_detector_scores_by_components(
        current_path, detectors, dataset_info22, arch, ds
    )

    row_data = [
        ds.replace("_", "\\_"),
        format_accuracy(orig_single, trans_single)[0],
        format_accuracy(orig_aug, trans_aug)[0],
        format_accuracy(orig_aug_val, trans_aug_val)[0],
        format_accuracy(orig_inv, trans_inv)[0],
        format_accuracy(orig_energy, trans_energy)[0],
        format_accuracy(orig_unsup, trans_unsup)[0],
    ]
    latex_lines.append(" & ".join(row_data) + " \\\\")

latex_lines.append("\\bottomrule")
latex_lines.append("\\end{tabular}")
latex_lines.append("\\end{table}")

# Table 2: Transformed Test Set
latex_lines.append("")
latex_lines.append("\\begin{table}[htbp]")
latex_lines.append("\\centering")
latex_lines.append("\\caption{Accuracy on Transformed Test Set (\\%)}")
latex_lines.append("\\label{tab:transformed_test}")
latex_lines.append("\\begin{tabular}{l|cccccc}")
latex_lines.append("\\toprule")
latex_lines.append("Dataset & Single & Augmented & Aug+Val & Invariant & Energy & Best Unsup. \\\\")
latex_lines.append("\\midrule")

for ds in all_datasets:
    arch = default_architecutre_mapping[ds]
    dataset_info22 = get_dataset_info(ds)
    trans_name = dataset_info22.transform_seq_name

    orig_single, trans_single = load_result(ds, arch, trans_name, "augmented_single_sample")
    orig_aug, trans_aug = load_result(ds, arch, trans_name, "augmented")
    orig_aug_val, trans_aug_val = load_result(ds, arch, trans_name, "augmented_val")

    if ds in ["mnist", "bigger_mnist", "emnist", "bigger_emnist", "coil100"]:
        orig_inv, trans_inv = load_result(ds, arch, trans_name, "escnn")
    elif ds == "tu_berlin":
        orig_inv, trans_inv = load_result(ds, arch, trans_name, "pca")
    elif ds == "modelnet10":
        orig_inv, trans_inv = load_result(ds, arch, trans_name, "pca_then_norm_randomize_sorted_no_aug")
    else:
        orig_inv, trans_inv = None, None

    orig_energy, trans_energy = load_energy_learned(dataset_info22, trans_name, current_path, ds, arch)
    if orig_energy is not None:
        orig_energy = orig_energy["accuracy_mean"]
        trans_energy = trans_energy["accuracy_mean"]

    unsupervised_best_name, orig_unsup, trans_unsup = load_best_detector_scores_by_components(
        current_path, detectors, dataset_info22, arch, ds
    )

    row_data = [
        ds.replace("_", "\\_"),
        format_accuracy(orig_single, trans_single)[1],
        format_accuracy(orig_aug, trans_aug)[1],
        format_accuracy(orig_aug_val, trans_aug_val)[1],
        format_accuracy(orig_inv, trans_inv)[1],
        format_accuracy(orig_energy, trans_energy)[1],
        format_accuracy(orig_unsup, trans_unsup)[1],
    ]
    latex_lines.append(" & ".join(row_data) + " \\\\")

latex_lines.append("\\bottomrule")
latex_lines.append("\\end{tabular}")
latex_lines.append("\\end{table}")


# Print LaTeX code
latex_output = "\n".join(latex_lines)
print(latex_output)

# Save to file
output_path = os.path.join(results_dir_path, "comparison_tables.tex")
with open(output_path, 'w') as f:
    f.write(latex_output)

print(f"\n\nLaTeX tables saved to: {output_path}")